In [1]:
import os
import pickle
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostClassifier, Pool
import warnings
import sys
from sklearn.preprocessing import LabelEncoder

import consts

warnings.filterwarnings('ignore')


In [2]:
RESULT_PREPROCESSED_PATH = r'C:\Users\Egor\Desktop\study\diploma\project\dataset_preprocessed'
class CFG:
    train = True
    output_dir = "./output"
    exp = "120"
    seed = 42
    fold = 5
    used_fold = [0, 1, 2, 3, 4]
    val_start_date = '2020-09-16'
    catboost_params = {
        'loss_function': 'Logloss',
        'learning_rate': 0.02,
        'max_depth': 6,
        'random_seed': seed,
        'thread_count': 4,
        'task_type': 'GPU',
        'scale_pos_weight': 100,
        'iterations': 5000,
        'early_stopping_rounds': 100,
        'verbose': 200,
        'fold_permutation_block': 32,
        'gpu_cat_features_storage': 'CpuPinnedMemory',
        "train_dir": "./output"
    }


In [3]:
def reduce_mem_usage(df):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print(f'Memory: {start_mem:.2f} Mb -> {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

def load_data(data_dir):
    print("Загрузка данных...")
    transactions = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "transactions.csv")))
    articles = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "articles.csv")))
    customers = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "customers.csv")))
    transactions = transactions[transactions['t_dat'] > '2020-01-01']
    le = LabelEncoder()
    all_customers = pd.concat([transactions['customer_id'], customers['customer_id']]).unique()
    le.fit(all_customers)
    transactions['customer_id'] = le.transform(transactions['customer_id']).astype('int32')
    customers['customer_id'] = le.transform(customers['customer_id']).astype('int32')

    transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
    transactions['day'] = (transactions['t_dat'].max() - transactions['t_dat']).dt.days.astype('int16')
    transactions['week'] = (transactions['day'] // 7).astype('int8')
    return transactions, articles, customers



In [4]:
def mapk(actual, predicted, k=12):
    def apk(a, p, k):
        if len(p) > k:
            p = p[:k]
        score = 0.0
        hits = 0.0
        for i, pred in enumerate(p):
            if pred in a and pred not in p[:i]:
                hits += 1.0
                score += hits / (i + 1.0)
        return score / min(len(a), k) if a else 0.0

    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

In [5]:

def save_preprocessor(obj, path):
    joblib.dump(obj, path)

In [6]:

def make_candidate_df(transactions, articles, customers, target_df, val_start_date = '2020-01-01', train=False):
    if train:
        top300 = target_df.groupby('article_id')['customer_id'].nunique().sort_values(ascending=False).head(300).index
    else:
        top300 = transactions.query(f"t_dat >= '{val_start_date}'").groupby('article_id')[
            'customer_id'].nunique().sort_values(ascending=False).head(300).index

    if train:
        customers_list = target_df['customer_id'].unique()
    else:
        customers_list = target_df['customer_id'].unique()
    # Создаём все комбинации с топ-300 товарами
    df = pd.DataFrame({'customer_id': customers_list})
    df['tmp'] = 0
    tmp2 = pd.DataFrame({'article_id': top300, 'tmp': 0})
    df = pd.merge(df, tmp2, on='tmp', how='outer').drop('tmp', axis=1)

    # Шаг 2: товары с тем же product_code, что уже купленные пользователем
    user_products = transactions[['customer_id', 'article_id']].drop_duplicates()
    user_products = user_products[user_products['customer_id'].isin(customers_list)]
    user_products = pd.merge(user_products, articles[['article_id', 'product_code']], on='article_id', how='left')
    user_products = user_products.drop_duplicates(['customer_id', 'product_code'])[['customer_id', 'product_code']]

    # Все товары, имеющие такие product_code
    articles_with_codes = articles[articles['product_code'].isin(user_products['product_code'].unique())][
        ['article_id', 'product_code']]
    extra = pd.merge(user_products, articles_with_codes, on='product_code', how='outer')[['customer_id', 'article_id']]
    extra = extra.dropna(subset=['customer_id', 'article_id']).astype({'customer_id': 'int32', 'article_id': 'int32'})

    # Объединяем и убираем дубликаты
    df = pd.concat([df, extra]).drop_duplicates(['customer_id', 'article_id']).reset_index(drop=True)
    df = df.merge(customers, on='customer_id', how='left')
    if train:
        # Добавляем метку target (1 если покупка в целевой период)
        target_pairs = target_df[['customer_id', 'article_id']].drop_duplicates()
        target_pairs['target'] = 1
        df = pd.merge(df, target_pairs, on=['customer_id', 'article_id'], how='left')
        df['target'] = df['target'].fillna(0).astype('int8')
    else:
        df['target'] = 0

    return df

def add_article_features(df, articles):
    article_cols = ['article_id', 'product_code', 'product_type_no', 'graphical_appearance_no',
                    'colour_group_code', 'index_group_no', 'section_no', 'garment_group_no']
    articles_sub = articles[article_cols].copy()
    df = pd.merge(df, articles_sub, on='article_id', how='left')
    return df


def add_customer_features(df, customers):
    df = pd.merge(df, customers[['customer_id', 'age_group', 'club_member_status']], on='customer_id', how='left')
    le = LabelEncoder()
    df['club_member_status'] = le.fit_transform(df['club_member_status'])
    return df


def preprocess_customers(customers, output_dir):
    customers = customers[['customer_id', 'age', 'club_member_status']].copy()
    customers['age'] = customers['age'].fillna(customers['age'].median()).astype(int)
    customers['age_bin'] = pd.cut(customers['age'], bins=[0, 18, 25, 35, 45, 55, 65, 120],
                                  labels=[0, 1, 2, 3, 4, 5, 6]).astype(int)
    customers['club_member_status'] = customers['club_member_status'].fillna('unknown')
    le = LabelEncoder()
    customers['club_code'] = le.fit_transform(customers['club_member_status'])
    save_preprocessor(le, os.path.join(output_dir, 'club_encoder.pkl'))
    return customers[['customer_id', 'age_bin', 'club_code']], []



def repeat_features(transactions, df):
    rep = transactions.drop_duplicates(['customer_id', 'article_id', 't_dat']).groupby(
        ['customer_id', 'article_id']).size().reset_index(name='repeat_count')

    # По пользователю
    # user_stats = rep.groupby('customer_id')['repeat_count'].agg(['mean', 'max', 'std', 'count', 'sum']).reset_index()
    # user_stats.columns = ['customer_id', 'mean_repeat_by_cust', 'max_repeat_by_cust', 'std_repeat_by_cust',
    #                       'unique_items_by_cust', 'total_purchases_by_cust']
    # user_stats['repeat_rate_by_cust'] = (user_stats['total_purchases_by_cust'] - user_stats['unique_items_by_cust']) / \
    #                                     user_stats['total_purchases_by_cust']
    # # По товару
    # item_stats = rep.groupby('article_id')['repeat_count'].agg(['mean', 'max', 'std', 'count', 'sum']).reset_index()
    # item_stats.columns = ['article_id', 'mean_repeat_by_item', 'max_repeat_by_item', 'std_repeat_by_item',
    #                       'unique_customers_by_item', 'total_sales_by_item']
    # item_stats['repeat_rate_by_item'] = (item_stats['total_sales_by_item'] - item_stats['unique_customers_by_item']) / \
    #                                     item_stats['total_sales_by_item']

    # df = pd.merge(df, user_stats, on='customer_id', how='left')
    # df = pd.merge(df, item_stats, on='article_id', how='left')
    # Заполняем NaN
    for col in df.columns:
        if 'repeat' in col or 'unique' in col or 'total' in col:
            df[col] = df[col].fillna(0)
    return df


def last_purchase_date(transactions, df):
    # for col in ['article_id', 'product_code', 'product_type_no', 'graphical_appearance_no',
    #             'colour_group_code', 'index_group_no', 'section_no', 'garment_group_no']:
    #     if col not in df.columns or col not in transactions.columns:
    #         continue
    #     tmp = transactions.groupby(['customer_id', col])['day'].min().reset_index().rename(
    #         columns={'day': f'last_{col}'})
    #     df = pd.merge(df, tmp, on=['customer_id', col], how='left')
    #     df[f'last_{col}'] = df[f'last_{col}'].fillna(9999).astype('int16')
    return df


def weekly_count_features(transactions, df, weeks=5):
    # for col in ['article_id']:
    #     if col not in df.columns:
    #         continue
    #     for w in range(weeks):
    #         cnt = transactions[transactions['week'] == w].groupby(col)['customer_id'].nunique().reset_index().rename(
    #             columns={'customer_id': f'{col}_week{w}_count'})
    #         df = pd.merge(df, cnt, on=col, how='left')
    #         df[f'{col}_week{w}_count'] = df[f'{col}_week{w}_count'].fillna(0).astype('int16')
    #     # изменение между неделями
    #     df[f'{col}_week0_ratio'] = (df[f'{col}_week0_count'] / (df[f'{col}_week1_count'] + 1e-5)).astype('float32')
    return df


def daily_count_features(transactions, df, days=7):
    # for col in ['article_id']:
    #     if col not in df.columns:
    #         continue
    #     for d in range(days):
    #         cnt = transactions[transactions['day'] == d].groupby(col)['customer_id'].nunique().reset_index().rename(
    #             columns={'customer_id': f'{col}_day{d}_count'})
    #         df = pd.merge(df, cnt, on=col, how='left')
    #         df[f'{col}_day{d}_count'] = df[f'{col}_day{d}_count'].fillna(0).astype('int16')
    #     df[f'{col}_day0_ratio'] = (df[f'{col}_day0_count'] / (df[f'{col}_day1_count'] + 1e-5)).astype('float32')
    return df


def price_features(transactions, df):
    # user_price = transactions.groupby('customer_id')['price'].agg(['max', 'mean']).reset_index()
    # user_price.columns = ['customer_id', 'max_price_by_cust', 'mean_price_by_cust']
    # df = pd.merge(df, user_price, on='customer_id', how='left')
    # средняя цена товара за последние 4 недели
    # recent = transactions[transactions['week'] < 4].groupby('article_id')['price'].mean().reset_index().rename(
    #     columns={'price': 'mean_price_4w'})
    # df = pd.merge(df, recent, on='article_id', how='left')
    # df['higher_than_max'] = (df['mean_price_4w'] > df['max_price_by_cust']).astype('int8')
    # df['higher_than_mean'] = (df['mean_price_4w'] > df['mean_price_by_cust']).astype('int8')
    # df[['max_price_by_cust', 'mean_price_by_cust', 'mean_price_4w']] = df[
    #     ['max_price_by_cust', 'mean_price_by_cust', 'mean_price_4w']].fillna(0).astype('float32')
    return df


def channel_features(transactions, df):
    # for w in [999, 3, 0]:
    #     sub = transactions[transactions['week'] <= w] if w != 999 else transactions
    #     # по пользователю
    #     user_ch = sub.groupby('customer_id')['sales_channel_id'].agg(['mean', 'max']).reset_index()
    #     user_ch.columns = ['customer_id', f'mean_channel_{w}w_cust', f'max_channel_{w}w_cust']
    #     df = pd.merge(df, user_ch, on='customer_id', how='left')
    #     # по товару
    #     item_ch = sub.groupby('article_id')['sales_channel_id'].agg(['mean', 'max']).reset_index()
    #     item_ch.columns = ['article_id', f'mean_channel_{w}w_item', f'max_channel_{w}w_item']
    #     df = pd.merge(df, item_ch, on='article_id', how='left')
    #     # заполнение
    #     for col in [f'mean_channel_{w}w_cust', f'max_channel_{w}w_cust', f'mean_channel_{w}w_item',
    #                 f'max_channel_{w}w_item']:
    #         df[col] = df[col].fillna(0)
    return df



def preprocess_articles(articles, output_dir):
    cat_cols = ['product_type_name', 'graphical_appearance_name',
                'colour_group_name', 'perceived_colour_value_name',
                'department_name', 'index_name', 'section_name']
    articles = articles[['article_id'] + cat_cols].copy()
    for col in cat_cols:
        articles[col] = articles[col].fillna('unknown')
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        articles[f'{col}_code'] = le.fit_transform(articles[col])
        encoders[col] = le
    feature_cols = [f'{col}_code' for col in cat_cols]
    save_preprocessor(encoders, os.path.join(output_dir, 'article_encoders.pkl'))
    return articles[['article_id'] + feature_cols], feature_cols


def build_full_dataset(transactions, articles, customers, target_df, train=False, output_dir='./output'):
    print("Формирование пар (customer, article)...")
    df_copy = make_candidate_df(transactions, articles, customers, target_df, train=train)
    print(df_copy.info())
    for col in ['club_member_status', 'fashion_news_frequency', 'postal_code', 'age_group', 'recency_cat', 'frequency_cat', 'unique_articles_cat']:
        le = LabelEncoder()
        df_copy[col] = le.fit_transform(df_copy[col])
    # print("Добавление признаков товаров...")
    # df_copy = add_article_features(df_copy, articles)
    # print("Добавление признаков пользователей...")
    # df_copy = add_customer_features(df_copy, customers)
    # print("Статистика повторных покупок...")
    # df_copy = repeat_features(transactions, df_copy)
    # print("Дата последней покупки...")
    # df_copy = last_purchase_date(transactions, df_copy)
    # print("Недельные популярности...")
    # df_copy = weekly_count_features(transactions, df_copy)
    # print("Дневные популярности...")
    # df_copy = daily_count_features(transactions, df_copy)
    # print("Ценовые признаки...")
    # df_copy = price_features(transactions, df_copy)
    # print("Признаки каналов...")
    # df_copy = channel_features(transactions, df_copy)

    drop_cols = ['customer_id', 'article_id', 'target']
    if train:
        drop_cols.append('target')
    feature_cols = [c for c in df_copy.columns if c not in drop_cols]
    return df_copy, feature_cols

In [7]:
transactions, articles, customers = load_data(RESULT_PREPROCESSED_PATH)
target_df = transactions[transactions['t_dat'] >= CFG.val_start_date].reset_index(drop=True)
transactions = transactions[transactions['t_dat'] < CFG.val_start_date].reset_index(drop=True)

Загрузка данных...
Memory: 2637.23 Mb -> 879.08 Mb (66.7% reduction)
Memory: 44.29 Mb -> 19.53 Mb (55.9% reduction)
Memory: 303.55 Mb -> 150.47 Mb (50.4% reduction)


In [8]:

df, feature_cols = build_full_dataset(transactions, articles, customers, target_df, train=True)

Формирование пар (customer, article)...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39801132 entries, 0 to 39801131
Data columns (total 31 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   customer_id              int32  
 1   article_id               int32  
 2   FN                       int8   
 3   Active                   int8   
 4   club_member_status       object 
 5   fashion_news_frequency   object 
 6   age                      int8   
 7   postal_code              object 
 8   mean_channel_999w_cust   float16
 9   max_channel_999w_cust    float16
 10  mean_channel_3w_cust     float16
 11  max_channel_3w_cust      float16
 12  mean_channel_0w_cust     float16
 13  max_channel_0w_cust      float16
 14  age_group                object 
 15  recency                  float16
 16  frequency                float16
 17  monetary                 float16
 18  avg_basket               float16
 19  recency_cat              object 
 20  freq

In [9]:
print("Обучение CatBoost с кросс-валидацией...")
unique_customers = df['customer_id'].unique()
kf = KFold(n_splits=CFG.fold, shuffle=True, random_state=CFG.seed)
fold_assign = np.zeros(len(df), dtype=int)
for i, (_, val_idx) in enumerate(kf.split(unique_customers)):
    val_cust = unique_customers[val_idx]
    fold_assign[df['customer_id'].isin(val_cust)] = i

oof_pred = np.zeros(len(df))
scores = []
for fold in range(CFG.fold):
    if fold not in CFG.used_fold:
        continue
    print(f"Fold {fold}")
    tr_idx = fold_assign != fold
    va_idx = fold_assign == fold
    X_train = df.loc[tr_idx, feature_cols]
    y_train = df.loc[tr_idx, 'target']
    X_val = df.loc[va_idx, feature_cols]
    y_val = df.loc[va_idx, 'target']

    # Категориальные признаки (укажем колонки, которые являются категориями)
    cat_cols = [c for c in feature_cols if c in ['product_code', 'product_type_no', 'graphical_appearance_no',
                                                 'colour_group_code', 'index_group_no', 'section_no',
                                                 'garment_group_no']]
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)

    model = CatBoostClassifier(**CFG.catboost_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=True)
    joblib.dump(model, os.path.join(CFG.output_dir, CFG.exp, f'model_fold{fold}.pkl'))

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_pred[va_idx] = val_pred
    # Считаем MAP@12 на валидации
    val_df = df.loc[va_idx, ['customer_id', 'article_id', 'target']].copy()
    val_df['pred'] = val_pred
    val_group = val_df.sort_values('pred', ascending=False).groupby('customer_id')['article_id'].apply(
        list).reset_index()
    target_group = target_df.groupby('customer_id')['article_id'].apply(list).reset_index()
    val_group = pd.merge(val_group, target_group, on='customer_id', how='left')
    score = mapk(val_group['article_id_y'], val_group['article_id_x'], k=12)
    scores.append(score)
    print(f"Fold {fold} MAP@12 = {score:.6f}")
print(f"Средний MAP@12 по фолдам: {np.mean(scores):.6f}")

Обучение CatBoost с кросс-валидацией...
Fold 0
0:	learn: 0.6802198	test: 0.6803680	best: 0.6803680 (0)	total: 155ms	remaining: 12m 52s
1:	learn: 0.6678950	test: 0.6681501	best: 0.6681501 (1)	total: 290ms	remaining: 12m 4s
2:	learn: 0.6561260	test: 0.6565057	best: 0.6565057 (2)	total: 424ms	remaining: 11m 45s
3:	learn: 0.6449125	test: 0.6453902	best: 0.6453902 (3)	total: 555ms	remaining: 11m 33s
4:	learn: 0.6342208	test: 0.6348040	best: 0.6348040 (4)	total: 691ms	remaining: 11m 29s
5:	learn: 0.6240160	test: 0.6247044	best: 0.6247044 (5)	total: 822ms	remaining: 11m 24s
6:	learn: 0.6142654	test: 0.6150570	best: 0.6150570 (6)	total: 960ms	remaining: 11m 24s
7:	learn: 0.6049499	test: 0.6058400	best: 0.6058400 (7)	total: 1.09s	remaining: 11m 22s
8:	learn: 0.5960788	test: 0.5970669	best: 0.5970669 (8)	total: 1.23s	remaining: 11m 21s
9:	learn: 0.5876176	test: 0.5886962	best: 0.5886962 (9)	total: 1.36s	remaining: 11m 21s
10:	learn: 0.5795482	test: 0.5807185	best: 0.5807185 (10)	total: 1.5s	rema